<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-07/blob/main/hw_07_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 — Time Series and Semi-Structured Data for ML

## 1. Setup and NYC Taxi Data Loading

In [33]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [34]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

In [35]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1.1 NYC Taxi Data Loading

Dataset: NYC TLC Yellow Taxi Trip Records, March 2024.
Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [36]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(URL, parquet_path)

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

df = pd.read_parquet(parquet_path, columns=columns)

df = df[
    (df["tpep_pickup_datetime"] >= "2024-03-01") &
    (df["tpep_pickup_datetime"] < "2024-04-01")
]

df = df.sample(
    n=min(500_000, len(df)),
    random_state=42
).copy()

print("Sampled date range:")
print(df["tpep_pickup_datetime"].min())
print(df["tpep_pickup_datetime"].max())

Sampled date range:
2024-03-01 00:00:03
2024-03-31 23:59:56


In [37]:
rows_before = len(df)

df = df[
    (df["fare_amount"] > 0) &
    (df["fare_amount"] < 500) &
    (df["trip_distance"] > 0) &
    (df["trip_distance"] < 100) &
    (df["PULocationID"].notna())
].copy()

print("Before cleaning:", rows_before)
print("After cleaning:", len(df))
print("Removed rows:", rows_before - len(df))

Before cleaning: 500000
After cleaning: 480094
Removed rows: 19906


In [38]:
df.to_sql(
    "nyc_taxi_raw",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
)

pd.read_sql("""
    SELECT COUNT(*) AS n_rows
    FROM nyc_taxi_raw;
""", engine)

,n_rows
0,480094


## 1.2 Daily Time-Series Aggregation

Aggregate NYC Taxi trips by pickup zone and local NYC calendar day.

In [39]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_daily;

        CREATE TABLE t7_hw_taxi_daily AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
            COUNT(*)::INTEGER AS trip_count,
            AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
            SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
        FROM nyc_taxi_raw
        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
          AND "PULocationID" IS NOT NULL
        GROUP BY
            "PULocationID"::INTEGER,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

        ALTER TABLE t7_hw_taxi_daily
            ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """))

print("Daily aggregation created successfully.")

Daily aggregation created successfully.


In [40]:
# Validation
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily;
""", engine))

display(pd.read_sql("""
    SELECT *
    FROM t7_hw_taxi_daily
    ORDER BY pickup_zone_id, ts_day
    LIMIT 5;
""", engine))

,n_rows,n_zones,first_day,last_day,total_trips
0,5438,248,2024-03-01,2024-03-31,480094


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance
0,1,2024-03-01,1,105.0,0.04
1,1,2024-03-02,2,95.0,18.43
2,1,2024-03-08,1,105.0,0.04
3,1,2024-03-11,1,10.0,2.07
4,1,2024-03-14,1,29.6,2.93


## 1.3 Calendar Spine — Complete Daily Time Series

Generate all calendar days for every observed pickup zone.
Missing daily trip counts are filled with zero.

In [41]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_daily_spine;

        CREATE TABLE t7_hw_taxi_daily_spine AS
        WITH zones AS (
            SELECT DISTINCT pickup_zone_id
            FROM t7_hw_taxi_daily
        ),
        calendar AS (
            SELECT generate_series(
                DATE '2024-03-01',
                DATE '2024-03-31',
                INTERVAL '1 day'
            )::DATE AS ts_day
        )
        SELECT
            z.pickup_zone_id,
            c.ts_day,
            COALESCE(d.trip_count, 0)::INTEGER AS trip_count,
            d.avg_fare,
            COALESCE(d.total_distance, 0)::NUMERIC(12, 2) AS total_distance
        FROM zones z
        CROSS JOIN calendar c
        LEFT JOIN t7_hw_taxi_daily d
            ON d.pickup_zone_id = z.pickup_zone_id
           AND d.ts_day = c.ts_day;

        ALTER TABLE t7_hw_taxi_daily_spine
            ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """))

print("Calendar spine created successfully.")

Calendar spine created successfully.


In [42]:
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        COUNT(DISTINCT ts_day) AS n_days,
        COUNT(*) FILTER (WHERE trip_count = 0) AS zero_trip_days,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM t7_hw_taxi_daily_spine;
""", engine))

,n_rows,n_zones,n_days,zero_trip_days,first_day,last_day
0,7688,248,31,2250,2024-03-01,2024-03-31


In [43]:
display(pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily_spine
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 35;
""", engine))

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,NaN,NaN
1,1,2024-03-02,2,1.0,NaN,NaN
2,1,2024-03-03,0,2.0,NaN,NaN
3,1,2024-03-04,0,0.0,NaN,NaN
4,1,2024-03-05,0,0.0,NaN,NaN
5,1,2024-03-06,0,0.0,NaN,NaN
6,1,2024-03-07,0,0.0,NaN,NaN
7,1,2024-03-08,1,0.0,1.0,NaN
8,1,2024-03-09,0,1.0,2.0,NaN
9,1,2024-03-10,0,0.0,0.0,NaN


In [44]:
display(pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 30) OVER (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ) AS trips_lag_30d
    FROM t7_hw_taxi_daily_spine
    WHERE pickup_zone_id = 1
    ORDER BY ts_day DESC
    LIMIT 3;
""", engine))

,pickup_zone_id,ts_day,trip_count,trips_lag_30d
0,1,2024-03-31,0,1.0
1,1,2024-03-30,0,NaN
2,1,2024-03-29,0,NaN


In [45]:
display(pd.read_sql("""
    WITH lagged AS (
        SELECT
            pickup_zone_id,
            ts_day,
            LAG(trip_count, 30) OVER (
                PARTITION BY pickup_zone_id
                ORDER BY ts_day
            ) AS trips_lag_30d
        FROM t7_hw_taxi_daily_spine
    )
    SELECT
        COUNT(*) AS total_rows,
        COUNT(trips_lag_30d) AS rows_with_lag_30d,
        COUNT(DISTINCT pickup_zone_id)
            FILTER (WHERE trips_lag_30d IS NOT NULL) AS zones_with_lag_30d
    FROM lagged;
""", engine))

,total_rows,rows_with_lag_30d,zones_with_lag_30d
0,7688,248,248


### Task 1 — Data Loading and Preparation

The NYC Taxi dataset (March 2024) was downloaded in Parquet format. A reproducible random sample of 500,000 trips was selected and cleaned by removing invalid fares, distances, and missing pickup zone IDs.

The cleaned data (480,094 trips) was loaded into PostgreSQL and aggregated by pickup zone and calendar day, producing 5,438 daily records across 248 zones.

A **Calendar Spine** was created using `generate_series()` and `CROSS JOIN` to ensure that every pickup zone has a continuous daily time series. Missing dates were filled with zero trip counts, while unknown average fares remained `NULL`.

The final calendar-aligned dataset contains **7,688 rows (248 zones × 31 days)** and supports accurate calendar-based lag, rolling, and expanding feature calculations.

**Note:** Since the dataset is sampled, zero trip counts represent missing observations in the sample and do not necessarily indicate zero actual taxi demand.

## 2. Time-Series ML Feature Engineering

### Combined Feature View
Lag, rolling, and expanding features calculated per pickup zone using a complete calendar spine.

In [46]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
        WITH features_basic AS (
            SELECT
                pickup_zone_id,
                ts_day,
                trip_count,
                avg_fare,
                total_distance,

                LAG(trip_count, 1) OVER w_pid AS trips_lag_1d,
                LAG(trip_count, 7) OVER w_pid AS trips_lag_7d,
                LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

                AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
                COUNT(avg_fare) OVER w7 AS fare_days_7d,

                STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
                COUNT(avg_fare) OVER w30 AS fare_days_30d,

                SUM(trip_count) OVER we AS cumulative_trips,
                AVG(trip_count) OVER we AS expanding_avg_trips

            FROM t7_hw_taxi_daily_spine

            WINDOW
                w_pid AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day
                ),
                w7 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP
                    RANGE BETWEEN INTERVAL '6 days' PRECEDING
                              AND CURRENT ROW
                ),
                w30 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP
                    RANGE BETWEEN INTERVAL '29 days' PRECEDING
                              AND CURRENT ROW
                ),
                we AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day
                    ROWS BETWEEN UNBOUNDED PRECEDING
                             AND CURRENT ROW
                )
        )
        SELECT *
        FROM features_basic
        WHERE trips_lag_30d IS NOT NULL;
    """))

print("Feature view created successfully.")

Feature view created successfully.


In [47]:
#Validation
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        COUNT(*) FILTER (
            WHERE rolling_avg_fare_7d IS NOT NULL
        ) AS valid_rolling_avg,
        COUNT(*) FILTER (
            WHERE rolling_std_fare_30d IS NOT NULL
        ) AS valid_rolling_std
    FROM v_t7_hw_taxi_features;
""", engine))

display(pd.read_sql("""
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id
    LIMIT 10;
""", engine))

,n_rows,n_zones,first_day,last_day,valid_rolling_avg,valid_rolling_std
0,248,248,2024-03-31,2024-03-31,234,233


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,fare_days_7d,rolling_std_fare_30d,fare_days_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,NaN,0.00,0,1,1,NaN,0,36.149456,9,12,0.387097
1,2,2024-03-31,0,NaN,0.00,0,0,0,NaN,0,NaN,1,1,0.032258
2,3,2024-03-31,0,NaN,0.00,0,1,1,51.750000,4,13.525523,14,18,0.580645
3,4,2024-03-31,48,16.94,129.68,77,64,27,18.208571,7,2.161566,30,1004,32.387097
4,6,2024-03-31,0,NaN,0.00,0,0,0,85.600000,1,49.415410,3,3,0.096774
5,7,2024-03-31,18,22.59,89.95,15,14,5,22.465714,7,5.049312,30,298,9.612903
6,8,2024-03-31,0,NaN,0.00,0,0,0,NaN,0,NaN,1,1,0.032258
7,9,2024-03-31,0,NaN,0.00,1,1,0,14.255000,2,20.396997,7,7,0.225806
8,10,2024-03-31,5,75.85,67.30,2,5,12,58.257143,7,12.322941,30,162,5.225806
9,11,2024-03-31,0,NaN,0.00,0,0,0,37.000000,2,11.738678,10,15,0.483871


### Task 2 — Time-Series Feature Engineering Summary

Time-series ML features were generated using PostgreSQL window functions, partitioned by pickup zone. The features include 1-, 7-, and 30-day lags, rolling 7-day average fare, rolling 30-day fare volatility, and expanding cumulative trip statistics.

A calendar spine ensures consistent daily intervals, making lag features correspond to actual calendar days. The final feature view contains **248 records**, one per pickup zone for March 31, 2024.

Rolling average fares are available for 234 zones, while rolling volatility is available for 233 zones. Missing values occur when insufficient fare observations exist within the corresponding window.

For production forecasting, additional historical data and leakage-safe feature alignment would be required.